# 16 — Do earlier price movements in other districts help?

**Optional research after the completed 09b direct-history search.** Use the same verified Python 3.12 project kernel and **Restart Kernel and Run All**. Notebook 15 does not need to be rerun. No downloads, new packages or preparation reruns are required.

**Hypothesis:** price pressure in other districts may precede a local surge. Add six peer-pressure/support features while keeping the frozen 09b Optuna settings, 2024–2025 training, 10% target and exact evaluation rows unchanged. This is one feature experiment, **not another parameter search**.

Peers match **item code, unit, state and shop type**. The target district is excluded. Use only the previous completed week (t−1); momentum compares peer weeks t−2 and t−1. Require at least two other districts for price/change summaries. Build peer observations from the cleaned daily source, **not from outcome-eligible modelling rows**. There is no national-pool fallback or future-value filling.

Compare all three expanding folds first, then January–March 2026 once. These already-inspected periods provide exploratory evidence, not independent final testing. **0.70 AP is an aspiration, not a promised result or a reason to keep searching until it appears.** No April–June test or registered November outcome is read. Dashboard/model pointers remain unchanged. Results save separately under the active run's `research/16_cross_district_signals/`.

**Availability limit:** a one-week lag is a conservative design choice, not proof that every historical record/lookup was published on time. The data remain retrospective, with unresolved source prices and changing monitored shops.

## 1. Verify the completed longer-history control

Use `optuna` from the latest completed 09b run, not whichever model happens to have the largest validation score. Restore matching inputs if verification fails; do not remove a hash guard. The saved settings are fixed before any new fit.

In [ ]:
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open this notebook inside the FYP repository.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import ast
import hashlib
import json
from datetime import datetime, timezone
from importlib import metadata, reload
from time import perf_counter
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             precision_score, recall_score, f1_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMClassifier
import pricecatcher_config as cfg
cfg = reload(cfg)

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

history_runs = sorted((cfg.PROJECT_ROOT / "09b_longer_history_lightgbm").glob("*/tuned/study_manifest.json"), reverse=True)
if not history_runs:
    raise FileNotFoundError("Complete 09b_longer_history_lightgbm.ipynb first.")
control_dir = history_runs[0].parent.parent / "optuna"
if not (control_dir / "study_manifest.json").is_file():
    raise FileNotFoundError("The latest 09b run needs its completed direct-history Optuna result.")
parent = cfg.verify_saved_study(control_dir)
selection = parent["selection"]
if selection["family"] != "LightGBM" or selection["training_start"] != "2024-01":
    raise ValueError("Expected the completed longer-history LightGBM control.")
if sys.version_info[:2] != tuple(map(int, parent["python_version"].split(".")[:2])):
    raise ValueError("Use the same Python major/minor version as 09b.")
for package in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]:
    if metadata.version(package) != parent["package_versions"][package]:
        raise ValueError(f"Use the saved 09b {package} version for this controlled comparison.")

FOLDS = ["fold_1", "fold_2", "fold_3"]
folders = {name: cfg.PROJECT_ROOT / "experiments" / name / "05_modelling" for name in FOLDS + ["holdout"]}
settings_by_fold = {name: json.loads((folder / "experiment_settings.json").read_text()) for name, folder in folders.items()}
settings = settings_by_fold["holdout"]
if any(settings["preparation"][key] != value for key, value in cfg.PREPARATION_SETTINGS.items()):
    raise ValueError("Current configuration differs from the saved preparation. Restore its matching configuration.")
FEATURES, NUMERIC, CATEGORICAL = (settings[key] for key in ["feature_columns", "numeric_features", "categorical_features"])
if FEATURES != selection["feature_columns"] or settings["surge_threshold_pct"] != 10:
    raise ValueError("Keep the control feature list and 10% definition unchanged.")
MODEL_PARAMETERS = {**selection["fixed_parameters"], **selection["parameters"],
                    "random_state": selection["seed"], "n_jobs": selection["n_jobs"]}
control_model = joblib.load(control_dir / "lightgbm.joblib")
fitted_parameters = control_model.named_steps["model"].get_params()
if any(fitted_parameters[key] != value for key, value in MODEL_PARAMETERS.items()):
    raise ValueError("Declared model settings differ from the saved control model.")
parent_manifest_sha256 = file_hash(control_dir / "study_manifest.json")
configuration_sha256 = file_hash(Path(cfg.__file__))
print("Active run:", cfg.PROJECT_ROOT)
print("Frozen control:", control_dir)
print("Parameters (unchanged):", MODEL_PARAMETERS)

## 2. Build label-free weekly peer observations and six features

For a target week t, use other districts' observations in t−1, matched by item/unit/state/shop type. Each peer district contributes one weekly median, regardless of its monitoring frequency. Retain only peer weeks with the original minimum record/premise support. A peer's percentage change additionally requires supported observations in the exact preceding week. No following-week price, target label or future eligibility is used.

| Feature | Meaning |
|---|---|
| `peer_change_median_pct_1w` | Median percentage change among other districts with supported t−2 → t−1 changes |
| `peer_surge_share_pct_1w` | Percentage of those districts whose **past** increase reached the existing 10% definition |
| `price_vs_peer_median_pct_1w` | Local t−1 price relative to the median price of other districts in t−1 |
| `peer_districts_1w` | Other districts supporting the price pool |
| `peer_change_districts_1w` | Other districts supporting the change pool |
| `peer_records_1w` | Observed records contributing to the other-district price pool |

Require two peers for each corresponding price/change summary. Support counts stay visible even when summaries are missing. Missing signals are imputed only inside each training-fitted pipeline. The supported target population is never reduced to improve AP. Unit-incompatible items, other states/shop types and the target district cannot enter the pool.

In [ ]:
MIN_PEER_DISTRICTS = 2
EXTRA_FEATURES = ["peer_change_median_pct_1w", "peer_surge_share_pct_1w", "price_vs_peer_median_pct_1w",
                  "peer_districts_1w", "peer_change_districts_1w", "peer_records_1w"]
PEER_KEYS = ["item_code", "unit", "state", "premise_type", "peer_week"]
FORMULAS = {
    "peer_change_median_pct_1w": "median of other districts' supported t-2 to t-1 percentage changes",
    "peer_surge_share_pct_1w": "100 * fraction of supported other-district past changes >= 10% (saved rounding tolerance)",
    "price_vs_peer_median_pct_1w": "100 * (local lag_1w_price / median(other-district t-1 medians) - 1)",
    "peer_districts_1w": "number of supported other districts at t-1",
    "peer_change_districts_1w": "number of those districts with supported exact-week changes",
    "peer_records_1w": "sum of observed records in the other-district t-1 price pool",
}
peer_support_rows = []

def build_peer_panel(name, source_end):
    """Rebuild only observed weekly prices up to the latest permitted peer day."""
    policy = settings_by_fold[name]["preparation"]
    stage3 = folders[name].parent / "03_cleaned"
    cleaning = json.loads((stage3 / "cleaning_summary.json").read_text())
    columns = cfg.GROUP_KEYS + ["date", "premise_code", "price"]
    daily = pd.read_parquet(stage3 / "daily_prices.parquet", columns=columns,
                            filters=[("date", "<=", source_end)])
    daily["date"] = pd.to_datetime(daily.date)
    history = earlier_source.loc[earlier_source.price.notna() & np.isfinite(earlier_source.price)
                                & earlier_source.price.gt(0)].copy()
    history = history.loc[history[cfg.GROUP_KEYS].notna().all(axis=1)]
    if policy["remove_borong"]:
        history = history.loc[~history.premise_type.eq("Borong")]
    history = history.loc[~history.item_code.isin(cleaning["excluded_item_codes"])]
    daily = pd.concat([history, daily], ignore_index=True)
    daily = daily.loc[daily.date.le(source_end)].copy()
    if daily.duplicated(["date", "premise_code", "item_code"]).any():
        raise ValueError("Duplicate peer source keys need investigation.")
    daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
    local_keys = ["item_code", "unit", "state", "district", "premise_type", "week_start"]
    weekly = daily.groupby(local_keys, sort=False).agg(price=("price", "median"),
        records=("price", "size"), premises=("premise_code", "nunique")).reset_index()
    weekly = weekly.loc[weekly.week_start.ge(pd.Timestamp("2024-01-01"))
        & (weekly.week_start + pd.Timedelta(days=6)).le(source_end)].copy()
    supported = weekly.records.ge(policy["minimum_records"]) & weekly.premises.ge(policy["minimum_premises"])
    weekly = weekly.loc[supported].copy()
    previous = weekly[local_keys + ["price"]].rename(columns={"price": "previous_price"})
    previous["week_start"] += pd.Timedelta(days=7)
    weekly = weekly.merge(previous, on=local_keys, how="left", validate="one_to_one")
    weekly["change_pct"] = 100 * (weekly.price / weekly.previous_price - 1)
    return weekly.rename(columns={"week_start": "peer_week"})[PEER_KEYS + ["district", "price", "change_pct", "records"]]

def add_peer_features(frame, panel):
    """Exclude the target district; use an exact t-1 lookup without filling gaps."""
    result = frame.copy()
    requests = result[["item_code", "unit", "state", "premise_type", "district", "week_start"]].copy()
    requests["peer_week"] = requests.week_start - pd.Timedelta(days=7)
    values = np.full((len(result), len(EXTRA_FEATURES)), np.nan)
    values[:, 3:] = 0  # No observed peers means zero support, not a zero price change.
    groups = panel.groupby(PEER_KEYS, sort=False).indices
    peer_district = panel.district.to_numpy()
    peer_price = panel.price.to_numpy(dtype=float)
    peer_change = panel.change_pct.to_numpy(dtype=float)
    peer_records = panel.records.to_numpy(dtype=float)
    target_district = requests.district.to_numpy()
    local_previous = result.lag_1w_price.to_numpy(dtype=float)
    tolerance = settings_by_fold["holdout"]["preparation"].get("label_tolerance_pct", 0.0)
    for group, targets in requests.groupby(PEER_KEYS, sort=False).indices.items():
        pool = groups.get(group)
        if pool is None:
            continue
        for position in targets:
            others = pool[peer_district[pool] != target_district[position]]
            changes = peer_change[others]
            changes = changes[np.isfinite(changes)]
            values[position, 3:] = [len(others), len(changes), peer_records[others].sum()]
            if len(others) >= MIN_PEER_DISTRICTS:
                values[position, 2] = 100 * (local_previous[position] / np.median(peer_price[others]) - 1)
            if len(changes) >= MIN_PEER_DISTRICTS:
                values[position, 0] = np.median(changes)
                values[position, 1] = 100 * cfg.surge_mask(changes, 10.0, tolerance).mean()
    if np.isinf(values).any():
        raise ValueError("Infinite peer feature; inspect retained prices.")
    result[EXTRA_FEATURES] = values
    return result

def attach_peer_context(name, training, evaluation):
    # Every requested t-1 week ends before its target current week starts.
    source_end = evaluation.week_start.max() - pd.Timedelta(days=1)
    panel = build_peer_panel(name, source_end)
    outputs = []
    for role, frame in [("training", training), ("evaluation", evaluation)]:
        extended = add_peer_features(frame, panel)
        pd.testing.assert_frame_equal(extended[frame.columns], frame)
        peer_support_rows.append({"fold": name, "role": role, "rows": len(extended),
            "rows_with_price_signal": int(extended.price_vs_peer_median_pct_1w.notna().sum()),
            "rows_with_change_signal": int(extended.peer_change_median_pct_1w.notna().sum()),
            "mean_other_districts": float(extended.peer_districts_1w.mean()),
            "latest_source_day": str(source_end.date())})
        outputs.append(extended)
    return outputs

def model_inputs(frame, enhanced=False):
    X = frame[FEATURES + (EXTRA_FEATURES if enhanced else [])].copy()
    for column in CATEGORICAL:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X

def fit_enhanced(training, evaluation):
    preprocess = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), NUMERIC + EXTRA_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL),
    ], sparse_threshold=1.0)
    model = Pipeline([("preprocess", preprocess), ("model", LGBMClassifier(**MODEL_PARAMETERS))])
    model.fit(model_inputs(training, enhanced=True), training.surge_label.astype(int))
    scores = model.predict_proba(model_inputs(evaluation, enhanced=True))[:, 1]
    if not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError("Expected finite model scores in [0, 1].")
    return model, scores

def evaluate(y, scores, cutoff):
    y = np.asarray(y, dtype=int)
    alerts = np.asarray(scores) >= cutoff
    tn, fp, fn, tp = confusion_matrix(y, alerts, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()), "average_precision":
        float(average_precision_score(y, scores)) if len(np.unique(y)) == 2 else np.nan,
        "cutoff": cutoff, "precision": precision_score(y, alerts, zero_division=0),
        "recall": recall_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "f1": f1_score(y, alerts, zero_division=0) if y.sum() else np.nan,
        "alerts": int(alerts.sum()), "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

## 3. Reuse 09b's verified training reconstruction

To avoid a second version of the historical-cleaning/weekly-label code, load **only the two named function definitions** from 09b. No notebook cells, downloads, tuning or deployment code are executed. As in 09b, original 2025 features/labels are reconstructed and checked before 2024 is added. The rebuilt counts must match this control's recorded populations.

The source functions, configuration and this notebook's calculation code are fingerprinted in the research manifest. The model inputs remain derived exclusively from the saved preparation and historical cache.

In [ ]:
history_notebook = REPOSITORY_ROOT / "09b_longer_history_lightgbm.ipynb"
source_nb = json.loads(history_notebook.read_text())
functions = []
for cell in source_nb["cells"]:
    if cell["cell_type"] == "code":
        tree = ast.parse("".join(cell["source"]))
        functions.extend(node for node in tree.body if isinstance(node, ast.FunctionDef)
                         and node.name in {"training_table", "prepare_training"})
if sorted(node.name for node in functions) != ["prepare_training", "training_table"]:
    raise ValueError("09b training helpers changed; review compatibility before this study.")
calculation = ast.Module(body=functions, type_ignores=[])
helper_sha256 = hashlib.sha256(ast.dump(calculation).encode()).hexdigest()
exec(compile(calculation, str(history_notebook), "exec"), globals())
helper_notebook_sha256 = file_hash(history_notebook)

additional_inputs = parent["additional_input_sha256"]
historical_paths = [cfg.resolve_input(path) for path in additional_inputs
                    if cfg.resolve_input(path).name == "selected_prices_2024.parquet"]
if len(historical_paths) != 1:
    raise ValueError("Expected the exact 2024 source snapshot recorded by 09b.")
earlier_source = pd.read_parquet(historical_paths[0], columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"])
earlier_source["date"] = pd.to_datetime(earlier_source.date)
if earlier_source.empty or not earlier_source.date.between("2024-01-01", "2024-12-31").all():
    raise ValueError("Supplementary source must contain only the recorded 2024 observations.")
features, keys = FEATURES, cfg.GROUP_KEYS + ["week_start"]
training_rows, cleaning_rows = [], []
expected_support = pd.read_csv(control_dir / "training_comparison.csv")

def matching_training(name):
    training, evaluation = prepare_training(name)
    for history, recorded in [("2025-only", training_rows[-2]), ("2024-added", training_rows[-1])]:
        expected = expected_support.loc[expected_support.fold.eq(name) & expected_support.history.eq(history)]
        if len(expected) != 1:
            raise ValueError(f"{name}: missing original support evidence.")
        for column in ["rows", "positives", "first_week", "last_outcome"]:
            if str(recorded[column]) != str(expected.iloc[0][column]):
                raise ValueError(f"{name}: rebuilt {history} {column} differs from 09b.")
    if training.surge_label.nunique() != 2 or evaluation.surge_label.nunique() != 2:
        raise ValueError("Both classes are needed for the overall AP comparison.")
    return training, evaluation

## 4. Compare expanding folds before later validation

Keep the frozen control parameters, seed and thread count. Rebuild the same longer-history training rows and outcome labels, then append the delayed peer features. Peers use all supported weekly source observations, including those without known next-week outcomes. Reuse the control's verified metrics. Do not choose a different peer delay, state scope or minimum support after seeing validation.

Report all three fold differences, the mean and feature coverage. A gain on these already-inspected folds is exploratory. Missing peer signals leave target rows in place; an all-missing signal is not evidence of improvement.

In [ ]:
control_folds = pd.read_csv(control_dir / "fold_comparison.csv")
control_folds = control_folds.loc[control_folds.method.eq(selection["comparison_method"])].copy()
if len(control_folds) != 3 or set(control_folds.fold) != set(FOLDS):
    raise ValueError("Expected exactly three recorded control folds.")
control_summary = pd.read_csv(control_dir / "fold_summary.csv").set_index("method").loc[selection["comparison_method"]]
np.testing.assert_allclose([control_summary.mean_ap, control_summary.std_ap],
    [control_folds.average_precision.mean(), control_folds.average_precision.std()], rtol=1e-10, atol=1e-12)
fold_rows = []
for name in FOLDS:
    training, evaluation = matching_training(name)
    training, evaluation = attach_peer_context(name, training, evaluation)
    started = perf_counter()
    model, scores = fit_enhanced(training, evaluation)
    original = control_folds.set_index("fold").loc[name]
    if int(original.rows) != len(evaluation) or int(original.positives) != int(evaluation.surge_label.sum()):
        raise ValueError("Control and enhanced fold evaluation populations differ.")
    fold_rows.append({"fold": name, "training_rows": len(training), "evaluation_rows": len(evaluation),
        "control_ap": float(original.average_precision),
        "enhanced_ap": float(average_precision_score(evaluation.surge_label, scores)),
        "fit_and_score_seconds": perf_counter() - started})
    del training, evaluation, model, scores

fold_comparison = pd.DataFrame(fold_rows)
fold_comparison["ap_change"] = fold_comparison.enhanced_ap - fold_comparison.control_ap
fold_summary = pd.DataFrame([
    {"method": "Frozen 09b Optuna control", "features": len(FEATURES),
     "mean_fold_ap": fold_comparison.control_ap.mean(), "std_fold_ap": fold_comparison.control_ap.std()},
    {"method": "Add delayed peer-price signals", "features": len(FEATURES) + len(EXTRA_FEATURES),
     "mean_fold_ap": fold_comparison.enhanced_ap.mean(), "std_fold_ap": fold_comparison.enhanced_ap.std()},
])
fold_gain = float(fold_comparison.ap_change.mean())
fold_conclusion = "Higher mean fold AP" if fold_gain > 0 else "No improvement in mean fold AP"
display(fold_comparison)
display(fold_summary)
print(fold_conclusion, "| difference:", round(fold_gain, 6))
print("Fold conclusion frozen before Jan–Mar validation; this is exploratory evidence.")
display(pd.DataFrame(peer_support_rows))

## 5. Evaluate January–March once, even if the folds declined

Use the saved control model and scores after verifying the matching identifiers and labels. Its fresh predictions must reproduce the saved scores. Only now rebuild holdout training and fit the enhanced model. The enhanced cutoff maximises validation F1 (highest cutoff breaks ties), exactly as 09b; these are apparent cutoff-selection metrics, not independent test results.

Report both models at the common 0.5 reference and their own selected cutoffs. Also report group counts so an aggregate gain cannot hide scarce positives. The fixed 10% event definition is unchanged. Reaching 0.70 only on folds does not establish 0.70 later-period performance.

In [ ]:
training, validation = matching_training("holdout")
training, validation = attach_peer_context("holdout", training, validation)
prediction_keys = ["item_code", "state", "district", "premise_type", "project_group",
                   "week_start", "next_week_end", "surge_label"]
identifiers = prediction_keys[:-1]
validation = validation.sort_values(identifiers).reset_index(drop=True)
control_predictions = pd.read_parquet(control_dir / "validation_predictions.parquet").sort_values(identifiers).reset_index(drop=True)
if validation.duplicated(identifiers).any() or control_predictions.duplicated(identifiers).any():
    raise ValueError("Duplicate evaluation identifiers.")
pd.testing.assert_frame_equal(validation[prediction_keys], control_predictions[prediction_keys])
control_scores = control_model.predict_proba(model_inputs(validation))[:, 1]
np.testing.assert_allclose(control_scores, control_predictions.model_score, rtol=1e-9, atol=1e-11)
enhanced_model, enhanced_scores = fit_enhanced(training, validation)
del training, control_model

p, r, cutoffs = precision_recall_curve(validation.surge_label, enhanced_scores)
f1 = np.divide(2*p[:-1]*r[:-1], p[:-1]+r[:-1], out=np.zeros(len(cutoffs)), where=(p[:-1]+r[:-1]) > 0)
enhanced_cutoff = float(cutoffs[np.flatnonzero(np.isclose(f1, f1.max(), rtol=0, atol=1e-12))[-1]])
score_sets = {"Frozen 09b Optuna control": (control_scores, float(selection["score_cutoff"])),
              "Add delayed peer-price signals": (enhanced_scores, enhanced_cutoff)}
validation_rows, group_rows = [], []
predictions = validation[prediction_keys].copy()
for method, (scores, cutoff) in score_sets.items():
    for role, threshold in [("common 0.5 reference", 0.5), ("apparent cutoff-selection performance", cutoff)]:
        validation_rows.append({"method": method, "role": role, **evaluate(validation.surge_label, scores, threshold)})
    for group, positions in validation.groupby("project_group", sort=True).indices.items():
        group_rows.append({"method": method, "project_group": group,
                           **evaluate(validation.iloc[positions].surge_label, scores[positions], cutoff)})
    prefix = "control" if method.startswith("Frozen") else "enhanced"
    predictions[prefix + "_score"] = scores
    predictions[prefix + "_alert"] = scores >= cutoff
validation_comparison = pd.DataFrame(validation_rows)
validation_groups = pd.DataFrame(group_rows)
display(validation_comparison)
display(validation_groups)
validation_gain = float(average_precision_score(validation.surge_label, enhanced_scores)
                        - average_precision_score(validation.surge_label, control_scores))
print("Later validation AP difference:", round(validation_gain, 6))
print("0.70 reached on mean folds:", bool(fold_comparison.enhanced_ap.mean() >= 0.70))
print("0.70 reached on later validation:", bool(average_precision_score(validation.surge_label, enhanced_scores) >= 0.70))
print("Neither period is an independent final test; no dashboard replacement is made.")

## 6. Save the complete research evidence

The model, formulas, peer coverage, row support, fold/validation/group metrics, scores, protocol and plot live together in one new study folder. A completed manifest is written last. No pointer, earlier experiment or source file is rewritten. Keep this research candidate separate from Notebook 10 and dashboard deployment.

In [ ]:
# Verify upstream bytes again before publishing a completed study.
cfg.verify_saved_study(control_dir)
if (file_hash(control_dir / "study_manifest.json") != parent_manifest_sha256
        or file_hash(history_notebook) != helper_notebook_sha256
        or file_hash(Path(cfg.__file__)) != configuration_sha256):
    raise ValueError("Parent study or calculation code changed during this experiment.")
this_notebook = REPOSITORY_ROOT / "research" / "16_cross_district_signals.ipynb"
notebook = json.loads(this_notebook.read_text())
code_sha256 = hashlib.sha256("\n".join("".join(c["source"]) for c in notebook["cells"]
                                       if c["cell_type"] == "code").encode()).hexdigest()
output = cfg.PROJECT_ROOT / "research" / "16_cross_district_signals" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
for name, table in {"fold_comparison.csv": fold_comparison, "fold_summary.csv": fold_summary,
    "validation_comparison.csv": validation_comparison, "validation_group_metrics.csv": validation_groups,
    "training_support.csv": pd.DataFrame(training_rows), "historical_cleaning.csv": pd.DataFrame(cleaning_rows), "peer_feature_support.csv": pd.DataFrame(peer_support_rows)}.items():
    table.to_csv(output / name, index=False)
predictions.to_parquet(output / "validation_predictions.parquet", index=False)
joblib.dump(enhanced_model, output / "research_model.joblib")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fold_comparison.set_index("fold")[["control_ap", "enhanced_ap"]].plot.bar(ax=axes[0], rot=0)
for method, (scores, _) in score_sets.items():
    precision, recall, _ = precision_recall_curve(validation.surge_label, scores)
    axes[1].plot(recall, precision, label=method)
axes[0].axhline(0.70, color="grey", linestyle=":", label="0.70 aspiration")
axes[0].set(ylabel="Average precision", ylim=(0, 1), title="Expanding folds (exploratory)")
axes[1].axhline(validation.surge_label.mean(), color="grey", linestyle=":", label="Positive fraction")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Jan–Mar validation (already inspected)")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(output / "comparison.png", dpi=150); plt.show()
protocol = {"status": "completed", "question": "delayed same-state other-district signals with fixed longer-history LightGBM settings",
    "python_version": sys.version.split()[0], "parameters": MODEL_PARAMETERS,
    "feature_columns": FEATURES + EXTRA_FEATURES, "extra_feature_formulas": FORMULAS,
    "peer_policy": {"keys": PEER_KEYS, "exclude_target_district": True, "lag_weeks": 1,
                    "minimum_other_districts": MIN_PEER_DISTRICTS, "uses_outcome_eligibility": False},
    "surge_threshold_pct": 10.0, "fold_conclusion": fold_conclusion,
    "mean_fold_ap_change": fold_gain, "validation_ap_change": validation_gain,
    "score_cutoff": enhanced_cutoff, "score_interpretation": "uncalibrated model score",
    "aspirational_ap": 0.70, "independent_test_evaluated": False, "dashboard_changed": False,
    "selection_rule": "fixed feature experiment; fold conclusion before later validation; no hyperparameter search",
    "validation_role": "already-inspected exploratory comparison and F1 cutoff selection",
    "parent_study": cfg.input_reference(control_dir / "study_manifest.json"),
    "parent_manifest_sha256": parent_manifest_sha256,
    "input_sha256": parent["input_sha256"], "additional_input_sha256": parent["additional_input_sha256"],
    "historical_helper_sha256": helper_sha256, "configuration_sha256": configuration_sha256,
    "research_code_sha256": code_sha256,
    "package_versions": {p: metadata.version(p) for p in ["lightgbm", "scikit-learn", "numpy", "pandas", "joblib"]},
    "output_sha256": {p.name: file_hash(p) for p in output.iterdir()}}
(output / "research_manifest.json").write_text(json.dumps(protocol, indent=2))
print("Saved:", output)
print("Share fold_summary.csv, validation_comparison.csv, validation_group_metrics.csv and peer_feature_support.csv for interpretation.")

## 7. Interpret improvement and its limits

- Compare fold and January–March AP with the **same frozen 09b Optuna control**. Do not compare a changed population to make scores look higher.
- Inspect `peer_feature_support.csv`: limited within-state peer coverage may make the idea weak or unusable in some places. Counts describe monitored districts/records, not market representativeness.
- Report group support, false alerts and missed surges. A selected-cutoff recall gain can come with more false alerts; AP and F1 need not move together.
- Positive results justify future independent evaluation, not automatic deployment or claims of causal price transmission. Common shocks, sampling changes and source errors may create apparent peer associations.
- If AP declines, preserve the result and stop this particular feature proposal. Do not adjust the delay/pool repeatedly against January–March or relabel the target to chase 0.70.

The current dashboard is unchanged. This is a six-feature extension of the original 22 inputs, **not a combination with the unsuccessful Notebook 15 ratios**. Integration would require a separate reviewed decision and matching serving/SHAP calculations.